<a href="https://colab.research.google.com/github/ziyad455/Coursezy/blob/main/fine_tuningNot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:


import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments

import torch


In [ ]:
# ============================================================================
# SECTION 1: CONFIGURATION
# ============================================================================

# Hugging Face Configuration
HF_TOKEN = ""  # Replace with your actual token
HF_USERNAME = "ZiyadTb"  # Replace with your HF username
DATASET_NAME = "your datadet"  # Replace with your dataset path
MODEL_NAME = "your modle"
OUTPUT_MODEL_NAME = ""  # Name for your fine-tuned model

# Training Configuration
MAX_SEQ_LENGTH = 512  # Maximum sequence length for training
NUM_EPOCHS = 3  # Number of training epochs
BATCH_SIZE = 4  # Batch size per device
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 4 * 4 = 16
LEARNING_RATE = 2e-4  # Learning rate for training
WARMUP_STEPS = 50  # Number of warmup steps

# LoRA Configuration
LORA_R = 16  # LoRA attention dimension
LORA_ALPHA = 32  # LoRA alpha parameter
LORA_DROPOUT = 0.05  # LoRA dropout

# Output directories
OUTPUT_DIR = "./results"
FINAL_MODEL_DIR = "./final_model"

print("=" * 80)
print("FINE-TUNING SCRIPT")
print("=" * 80)
print(f"Base Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_NAME}")
print(f"Output Model: {OUTPUT_MODEL_NAME}")
print("=" * 80)

In [6]:
# ============================================================================
# SECTION 2: LOGIN TO HUGGING FACE HUB
# ============================================================================

print("\n[1/8] Logging into Hugging Face Hub...")

from huggingface_hub import login

try:
    login(token=HF_TOKEN)
    print("✅ Successfully logged into Hugging Face Hub")
except Exception as e:
    print(f"❌ Failed to login: {e}")
    print("Please check your HF_TOKEN and try again")
    exit(1)

# ============================================================================
# SECTION 3: LOAD TOKENIZER
# ============================================================================

print("\n[2/8] Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN
)

# Set padding token (required for training)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Recommended for decoder-only models

print(f"✅ Tokenizer loaded successfully")
print(f"   Vocab size: {len(tokenizer)}")
print(f"   Pad token: {tokenizer.pad_token}")


[1/8] Logging into Hugging Face Hub...
✅ Successfully logged into Hugging Face Hub

[2/8] Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded successfully
   Vocab size: 151665
   Pad token: <|im_end|>


In [7]:
# ============================================================================
# SECTION 4: CONFIGURE 4-BIT QUANTIZATION
# ============================================================================

print("\n[3/8] Configuring 4-bit quantization...")

# BitsAndBytes configuration for 4-bit quantization
# This reduces memory usage significantly while maintaining model quality
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",  # Use NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
    bnb_4bit_use_double_quant=True,  # Enable nested quantization for more memory savings
)

print("✅ Quantization config created")
print(f"   Quantization type: 4-bit NF4")
print(f"   Compute dtype: float16")


[3/8] Configuring 4-bit quantization...
✅ Quantization config created
   Quantization type: 4-bit NF4
   Compute dtype: float16


In [8]:
# ============================================================================
# SECTION 5: LOAD BASE MODEL WITH QUANTIZATION
# ============================================================================

print("\n[4/8] Loading base model with 4-bit quantization...")
print("   (This may take a few minutes...)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",  # Automatically distribute model across available GPUs
    trust_remote_code=True,
)

# Prepare model for k-bit training (required for QLoRA)
model = prepare_model_for_kbit_training(model)

print("✅ Model loaded and prepared for training")
print(f"   Model size: ~{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")


[4/8] Loading base model with 4-bit quantization...
   (This may take a few minutes...)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded and prepared for training
   Model size: ~1.70B parameters


In [ ]:
# ============================================================================
# SECTION 6: CONFIGURE AND APPLY LORA
# ============================================================================

print("\n[5/8] Configuring and applying LoRA...")

# LoRA configuration for Qwen models
# Qwen uses different module names than Llama
lora_config = LoraConfig(
    r=LORA_R,  # Rank of the low-rank matrices
    lora_alpha=LORA_ALPHA,  # Scaling factor
    target_modules=[  # Qwen-specific module names
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = 100 * trainable_params / total_params

print("✅ LoRA applied successfully")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_percentage:.2f}%)")
print(f"   Total parameters: {total_params:,}")

In [ ]:
# ============================================================================
# SECTION 7: LOAD AND FORMAT DATASET
# ============================================================================
print("\n[6/8] Loading dataset from local file...")

# Load your local JSONL dataset
dataset = load_dataset(
    "json",  # Specify the file format
    data_files="your path",
    split="train"
)

print(f"✅ Dataset loaded successfully")
print(f"   Number of examples: {len(dataset)}")
print(f"   Dataset columns: {dataset.column_names}")

# Format dataset for instruction fine-tuning
def format_instruction(example):
    """
    Format each example into the Qwen 2.5 instruction format.

    Expected dataset format:
    - "input": The user's question
    - "output": The assistant's response

    Qwen 2.5 Instruct format uses ChatML:
    <|im_start|>system
    {system_message}<|im_end|>
    <|im_start|>user
    {user_message}<|im_end|>
    <|im_start|>assistant
    {assistant_message}<|im_end|>
    """
    system_message = """You are Coursezy Assistant, an expert AI helper for the Coursezy platform. Your role is to provide clear, accurate, and helpful guidance to coaches and students using the platform.

When answering questions:
- Give concise, well-structured responses that directly address the user's needs
- Always include the relevant HTML link when discussing platform features or actions
- Use a friendly, professional tone that makes coaches feel supported
- Break down complex processes into simple, actionable steps
- Anticipate follow-up questions and address them proactively
- If a feature has prerequisites or dependencies, mention them upfront

Format your responses with:
- Clear explanations in natural language
- Relevant links formatted as: [Feature Name](https://coursezy.com/path)
- Step-by-step instructions when appropriate
- Helpful tips or best practices when relevant

Remember: Your goal is to empower coaches to use Coursezy effectively and confidently."""

    text = f"""<|im_start|>system
{system_message}<|im_end|>
<|im_start|>user
{example['input']}<|im_end|>
<|im_start|>assistant
{example['output']}<|im_end|>"""

    return {"text": text}

# Apply formatting to dataset
formatted_dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)

print("✅ Dataset formatted for instruction tuning")
print(f"   Sample formatted text (first 200 chars):")
print(f"   {formatted_dataset[0]['text'][:200]}...")

In [12]:
# ============================================================================
# SECTION 8: CONFIGURE TRAINING ARGUMENTS
# ============================================================================

print("\n[7/8] Configuring training arguments...")

training_args = TrainingArguments(
    # Output configuration
    output_dir=OUTPUT_DIR,

    # Training hyperparameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,

    # Optimization settings
    optim="paged_adamw_8bit",  # Use 8-bit AdamW optimizer for memory efficiency
    fp16=True,  # Use mixed precision training

    # Logging and saving
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,  # Keep only 2 checkpoints to save disk space

    # Evaluation (optional - uncomment if you have a validation set)
    # evaluation_strategy="steps",
    # eval_steps=100,

    # Other settings
    max_grad_norm=1.0,  # Gradient clipping
    lr_scheduler_type="cosine",  # Learning rate scheduler
    report_to="none",  # Disable wandb/tensorboard (change if you want logging)
)

print("✅ Training arguments configured")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"   Learning rate: {LEARNING_RATE}")



[7/8] Configuring training arguments...
✅ Training arguments configured
   Epochs: 3
   Effective batch size: 16
   Learning rate: 0.0002


In [ ]:
training_args = TrainingArguments(
    # Output configuration
    output_dir="./results",

    # Training hyperparameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,

    # Optimization settings
    optim="paged_adamw_8bit",  # Use 8-bit AdamW optimizer for memory efficiency
    fp16=True,  # Use mixed precision training

    # Logging and saving
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,  # Keep only 2 checkpoints to save disk space

    # Evaluation (optional - uncomment if you have a validation set)
    # evaluation_strategy="steps",
    # eval_steps=100,

    # Other settings
    max_grad_norm=1.0,  # Gradient clipping
    lr_scheduler_type="cosine",  # Learning rate scheduler
    report_to="none",  # Disable wandb/tensorboard (change if you want logging)
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args
)

# Start training
trainer.train()

print("\n" + "=" * 80)
print("✅ Training completed successfully!")
print("=" * 80)

In [ ]:
# ============================================================================
# SECTION 10: MERGE LORA WEIGHTS AND SAVE MODEL
# ============================================================================

print("\n[9/10] Merging LoRA weights into base model...")

# Merge LoRA weights back into the base model
# This creates a single model file instead of base + adapter
model = model.merge_and_unload()

print("✅ LoRA weights merged successfully")

# Save the final model locally
print(f"\n[10/10] Saving final model to {FINAL_MODEL_DIR}...")

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("✅ Model saved locally")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "YOUR_USERNAME/YOUR_MODEL_NAME"  # 👈 change this

token = "YOUR_HF_TOKEN"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
model = AutoModelForCausalLM.from_pretrained(model_name, token=token)

prompt = "Where do I create a course?"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    inputs.input_ids,
    max_new_tokens=150,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
# ============================================================================
# SECTION 11: PUSH MODEL TO HUGGING FACE HUB
# ============================================================================

print("\n[11/11] Pushing model to Hugging Face Hub...")
print(f"   Repository: {HF_USERNAME}/{OUTPUT_MODEL_NAME}")

try:
    # Push model to Hub
    model.push_to_hub(
        f"{HF_USERNAME}/{OUTPUT_MODEL_NAME}",
        token=HF_TOKEN,
        private=False,  # Set to True if you want a private model
    )

    # Push tokenizer to Hub
    tokenizer.push_to_hub(
        f"{HF_USERNAME}/{OUTPUT_MODEL_NAME}",
        token=HF_TOKEN,
        private=False,
    )

    print("✅ Model successfully pushed to Hugging Face Hub!")
    print(f"   Model URL: https://huggingface.co/{HF_USERNAME}/{OUTPUT_MODEL_NAME}")

except Exception as e:
    print(f"❌ Failed to push model to Hub: {e}")
    print("   The model is still saved locally in:", FINAL_MODEL_DIR)

# ============================================================================
# COMPLETION MESSAGE AND NEXT STEPS
# ============================================================================

print("\n" + "=" * 80)
print(" FINE-TUNING COMPLETE!")
print("=" * 80)
print("\nYour Qwen model has been fine-tuned and pushed to Hugging Face Hub.")
print("\n📋 NEXT STEPS - DEPLOYING AS INFERENCE ENDPOINT:")
print("-" * 80)
print("\n1. Go to: https://huggingface.co/{}/{}/deploy".format(HF_USERNAME, OUTPUT_MODEL_NAME))
print("\n2. Click 'Deploy' → 'Inference Endpoints'")
print("\n3. Configure your endpoint:")
print("   - Choose GPU type (e.g., NVIDIA T4, A10G)")
print("   - Set instance type based on your needs")
print("   - Configure auto-scaling if needed")
print("\n4. Once deployed, you'll get an API endpoint URL")
print("\n5. Use the endpoint in your application via API calls:")
print("\n   Example API usage (NOT included in this script):")
print("   ```python")
print("   import requests")
print("   ")
print("   API_URL = 'https://your-endpoint.aws.endpoints.huggingface.cloud'")
print("   headers = {'Authorization': 'Bearer YOUR_HF_TOKEN'}")
print("   ")
print("   payload = {")
print("       'inputs': 'Where do I create a course?',")
print("       'parameters': {")
print("           'max_new_tokens': 150,")
print("           'temperature': 0.7,")
print("       }")
print("   }")
print("   ")
print("   response = requests.post(API_URL, headers=headers, json=payload)")
print("   print(response.json())")
print("   ```")
print("\n6. Integrate the API endpoint into your Laravel application")
print("\n📚 Documentation:")
print("   - Inference Endpoints: https://huggingface.co/docs/inference-endpoints")
print("   - API Reference: https://huggingface.co/docs/api-inference")
print("\n" + "=" * 80)
print("✨ Your Qwen model is ready for production use via API!")
print("=" * 80)